### Household Rosters
### _hhr.ipynb


Sarah Sullivan

Created: April 7, 2026 

Last Updated: June 15, 2026


This script inputs the file "_psid_long_matrix.dta," outputted by the script _psid.do in part II.

I do some datatype manipulation to create variables measuring changes between household rosters constructed in _psid.do. 

I then output the csv file "_hhr.csv" which gets merged back onto the file "_psid_long.dta" in part III of _psid.do

(You might be thinking, couldn't I have done this all in Stata? Yeah, I think I could've, but I didn't.)

In [1]:
# import packages
# version of pandas is 2.0.3, numpy is 2.0.3

import numpy as np
import pandas as pd

In [2]:
# read in data output from _psid.do part I step 32. 
output = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/_output"
df = pd.read_stata(output + "/_psid_long_matrix.dta", convert_categoricals=False)

In [3]:
# convert float variables to integers
floatvars = ["fam", "ID", "year", "age_", "fam_id_", "sample_indiv_N", "sample_indiv_A", "sample_indiv_B", "year_left_survey", "why_left_survey"]

for i in floatvars:
    df.fillna({i:0}, inplace=True)
    df[i] = df[i].astype(int)

In [4]:
# convert list-like strings of rosters to real lists
df['hhr'] = [[] for _ in range(len(df))]
df['ages'] = [[] for _ in range(len(df))]
df['rel'] = [[] for _ in range(len(df))]

df['siblings'] = [[] for _ in range(len(df))]
df['parents'] = [[] for _ in range(len(df))]
df['grandparents'] = [[] for _ in range(len(df))]

df['hhr'] = df['hhr_matrix'].apply(lambda x: x.split())
df['rel'] = df['rel_matrix'].apply(lambda x: x.split())
df['ages'] = df['age_matrix'].apply(lambda x: x.split())

df['siblings'] = df['sib_list'].apply(lambda x: x.split())
df['parents'] = df['par_list'].apply(lambda x: x.split())
df['grandparents'] = df['gpar_list'].apply(lambda x: x.split())


#### Using Matrix observations + own relationship to head + age observations

In [5]:
# within-person previous roster (ordered by year)
df = df.sort_values(["ID", "year"]).copy()

df['hhr_prev'] = df.groupby('ID')['hhr'].shift(1)
df['rel_prev'] = df.groupby('ID')['rel'].shift(1)
df['ages_prev'] = df.groupby('ID')['ages'].shift(1)

In [6]:
# fill missings with empty lists
df['hhr_prev'] = df['hhr_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['rel_prev'] = df['rel_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['ages_prev'] = df['ages_prev'].apply(lambda d: d if isinstance(d, list) else [])


In [7]:
# ids of who left and who came in each year (within person)
df["IDs_left"] = df.apply(
    lambda row: [x for x in row["hhr_prev"] if x not in row["hhr"]],
    axis=1
)
df["IDs_came"] = df.apply(
    lambda row: [x for x in row["hhr"] if x not in row["hhr_prev"]],
    axis=1
)

In [8]:
# flag first and last year of observation for each individual
df['is_first'] = ~df['ID'].duplicated(keep='first')
df['is_last'] = ~df['ID'].duplicated(keep='last')

In [9]:
df['IDs_came'] = df.apply(lambda row: [] if row['is_first'] else row['IDs_came'], axis=1)
df['IDs_left'] = df.apply(lambda row: [] if row['is_last'] else row['IDs_left'], axis=1)

In [10]:
# function for age of who left/came
def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [ages[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(ages)]  

In [11]:
# function for relationship of who left/came to individual
def get_rel_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    rel_prev = row['rel_prev'] if isinstance(row['rel_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [rel_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(rel_prev)]

def get_rel_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    rel = row['rel'] if isinstance(row['rel'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [rel[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(rel)]  

In [12]:
# apply fns above 

df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

df['rel_left'] = df.apply(get_rel_left, axis=1)
df['rel_came'] = df.apply(get_rel_came, axis=1)

In [13]:
# convert elements of ages_left and ages_came to list of integers
df['ages_left'] = df['ages_left'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])
df['ages_came'] = df['ages_came'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])

In [14]:
check1 = df.apply(lambda row: len(row['IDs_left']) == len(row['ages_left']) == len(row['rel_left']), axis=1).all()
check2 = df.apply(lambda row: len(row['IDs_came']) == len(row['ages_came']) == len(row['rel_came']), axis=1).all()
print(f"Check 1 (left): {check1}")
print(f"Check 2 (came): {check2}")

Check 1 (left): True
Check 2 (came): True


In [15]:
# add in relationship matrix information to identify who came / left

relationships_xy = {10: "Self", 20: "Legal Spouse", 22: "Partner", 30: "Child", 33: "Stepchild", 35: "Social child", 37: "Child in law", 38: "Foster child", 39: "Social child in law", 40: "Sibling", 43: "Step sibling", 45: "Social sibling", 47: "Sibling in law", 48: "Social sibling in law", 50: "Parent", 53: "Step parent", 55: "Social parent", 56: "Foster parent", 57: "Parent in law", 58: "Social parent in law", 60: "Grandchild", 61: "Step grandchild", 62: "Grandchild in law", 63: "Step great grandchild", 64: "Great grandchild in law", 65: "Great grandchild", 66: "Grandparent", 67: "Grandparent in law", 68: "Great grandparent", 69: "Great grandparent in law", 70: "Nephew or niece", 71: "Nephew or niece by marriage", 72: "Uncle or aunt", 73: "Uncle or aunt by marriage", 74: "Cousin", 75: "Cousin by marriage", 80: "Social grandchild", 81: "Social great grandchild", 82: "Social grandparent", 83: "Social great grandparent", 84: "Social nephew or niece", 85: "Social uncle or aunt", 86: "Social cousin", 87: "Step grandparent", 88: "Step great grandparent", 89: "Other foster relative", 94: "Unknown", 95: "Other relative", 96: "Other relative by marriage", 97: "Other social relative", 98: "Nonrelative"}

relationships_yx = {10: "Self", 20: "Legal Spouse", 22: "Partner", 30: "Parent", 33: "Stepparent", 35: "Social parent", 37: "Parent in law", 38: "Foster parent", 39: "Social parent in law", 40: "Sibling", 43: "Step sibling", 45: "Social sibling", 47: "Sibling in law", 48: "Social sibling in law", 50: "Child", 53: "Step child", 55: "Social child", 56: "Foster child", 57: "Child in law", 58: "Social child in law", 60: "Grandparent", 61: "Step grandparent", 62: "Grandparent in law", 63: "Step great grandparent", 64: "Great grandparent in law", 65: "Great grandparent", 66: "Grandchild", 67: "Grandchild in law", 68: "Great grandchild", 69: "Great grandchild in law", 70: "Aunt or uncle", 71: "Aunt or uncle by marriage", 72: "Nephew or niece", 73: "Nephew or niece by marriage", 74: "Cousin", 75: "Cousin by marriage", 80: "Social grandparent", 81: "Social great grandparent", 82: "Social grandchild", 83: "Social great grandchild", 84: "Social uncle or aunt", 85: "Social nephew or niece", 86: "Social cousin", 87: "Step grandchild", 88: "Step great grandchild", 89: "Other foster relative", 94: "Unknown", 95: "Other relative", 96: "Other relative by marriage", 97: "Other social relative", 98: "Nonrelative"}

In [16]:
# sort by adult vs. child came or left. 

df['adult_came'] = df['ages_came'].apply(lambda ages: isinstance(ages, list) and any(age in range(18, 999) for age in ages))
df['adult_left'] = df['ages_left'].apply(lambda ages: isinstance(ages, list) and any(age in range(18, 999) for age in ages))

df['child_came'] = df['ages_came'].apply(lambda ages: isinstance(ages, list) and any(age in range(0, 18) for age in ages))
df['child_left'] = df['ages_left'].apply(lambda ages: isinstance(ages, list) and any(age in range(0, 18) for age in ages))

df['unknown_came'] = df['ages_came'].apply(lambda ages: isinstance(ages, list) and any(age == 999 for age in ages))
df['unknown_left'] = df['ages_left'].apply(lambda ages: isinstance(ages, list) and any(age == 999 for age in ages))


In [17]:
# relationships of who came/left to individual
df['rel_left_desc'] = df.apply(lambda row: [relationships_yx.get(int(rel)) for rel in row['rel_left']] if isinstance(row['rel_left'], list) else [], axis=1)
df['rel_came_desc'] = df.apply(lambda row: [relationships_yx.get(int(rel)) for rel in row['rel_came']] if isinstance(row['rel_came'], list) else [], axis=1)

In [18]:
# make lists of IDs. 

# parents
any_parent = ["30", "33", "35", "37", "38", "39"]
nonbio_parent = ["33", "35", "37", "38", "39"]

bio_parent = ["30"]
step_parent = ["33"]
social_parent = ["35"]
in_law_parent = ["37"]
foster_parent = ["38"]
social_in_law_parent = ["39"]

# (great) grandparents
any_gpar_ggpar = ["60", "61", "62", "80", "65", "63", "64", "81"]
any_gpar = ["60", "61", "62", "80"]
any_ggpar = ["65", "63", "64", "81"]
bio_grandparent = ["60"]
step_grandparent = ["61"]
in_law_grandparent = ["62"]
social_grandparent = ["80"]

bio_greatgrandparent = ["65"]
step_greatgrandparent = ["63"]
inlaw_greatgrandparent = ["64"]
social_greatgrandparent = ["81"]

# Aunt / uncle
any_auntuncle = ["70", "71", "84"]
bio_auntuncle = ["70"]
marriage_auntuncle = ["71"]
social_auntuncle = ["84"]

# Siblings
any_sibling = ["40", "43", "45", "47", "48"]

bio_sibling = ["40"]
step_sibling = ["43"]
social_sibling = ["45"]
in_law_sibling= ["47"]
social_in_law_sibling = ["48"]

# Cousins 
any_cousin = ["74", "75", "86"]

cousin = ["74"]
marriage_cousin = ["75"]
social_cousin = ["86"]

# niece/nephew
any_nephniece = ["72", "73", "85"]
bio_nephniece = ["72"]
marriage_nephniece = ["73"]
social_nephniece = ["85"]


# children --> should approx 0
any_ownchild = ["50", "53", "55", "56", "57", "58"]
bio_child = ["50"]
step_child = ["53"]
social_child = ["55"]
foster_child = ["56"]
in_law_child = ["57"]
social_in_law_child = ["58"]


# Grandchild / greatgrandchild --> should all be missing
any_grandchild = ["60", "61", "62", "80"]
any_greatgrandchild = ["65", "63", "64", "81"]
any_ggchild = ["60", "61", "62", "80", "65", "63", "64", "81"]
bio_grandchild = ["66"]
step_grandchild = ["87"]
inlaw_grandchild = ["67"]
social_grandchild = ["82"]
bio_greatgrandchild = ["68"]
step_greatgrandchild = ["88"]
inlaw_greatgrandchild = ["69"]
social_greatgrandchild = ["83"]

# unknown 
unknown = ["94"]

# foster
foster_other = ["89"]

# other relative
any_relativeother = ["95", "96", "97"]
bio_relativeother = ["95"]
marriage_relativeother = ["96"]
social_relativeother = ["97"]

# nonrelative
nonrelative = ["98"]

# spouse/partner

spouse_partner = ["20", "22"]
spouse = ["20"]
partner = ["22"]


In [19]:
# parents
new_df = {}

new_df['parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_parent for rel in rels))
new_df['parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_parent for rel in rels))

new_df['nonbio_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in nonbio_parent for rel in rels))
new_df['nonbio_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in nonbio_parent for rel in rels))

new_df['bio_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_parent for rel in rels))
new_df['bio_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_parent for rel in rels))

new_df['step_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_parent for rel in rels))
new_df['step_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_parent for rel in rels))

new_df['social_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_parent for rel in rels))
new_df['social_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_parent for rel in rels))

new_df['in_law_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_parent for rel in rels))
new_df['in_law_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_parent for rel in rels))

new_df['foster_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_parent for rel in rels))
new_df['foster_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_parent for rel in rels))

new_df['social_in_law_parent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_parent for rel in rels))
new_df['social_in_law_parent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_parent for rel in rels))


# (great) grandparents
new_df['any_gpar_ggpar_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_gpar_ggpar for rel in rels))
new_df['any_gpar_ggpar_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_gpar_ggpar for rel in rels))

new_df['any_gpar_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_gpar for rel in rels))
new_df['any_gpar_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_gpar for rel in rels))
new_df['any_ggpar_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ggpar for rel in rels))
new_df['any_ggpar_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ggpar for rel in rels))

new_df['bio_grandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_grandparent for rel in rels))
new_df['bio_grandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_grandparent for rel in rels))
new_df['step_grandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_grandparent for rel in rels))
new_df['step_grandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_grandparent for rel in rels))
new_df['in_law_grandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_grandparent for rel in rels))
new_df['in_law_grandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_grandparent for rel in rels))
new_df['social_grandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_grandparent for rel in rels))
new_df['social_grandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_grandparent for rel in rels))


new_df['bio_greatgrandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_greatgrandparent for rel in rels))
new_df['bio_greatgrandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_greatgrandparent for rel in rels))
new_df['step_greatgrandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_greatgrandparent for rel in rels))
new_df['step_greatgrandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_greatgrandparent for rel in rels))
new_df['inlaw_greatgrandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_greatgrandparent for rel in rels))
new_df['inlaw_greatgrandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_greatgrandparent for rel in rels))
new_df['social_greatgrandparent_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_greatgrandparent for rel in rels)) 
new_df['social_greatgrandparent_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_greatgrandparent for rel in rels))


# Aunt / uncle
new_df['any_auntuncle_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_auntuncle for rel in rels))
new_df['any_auntuncle_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_auntuncle for rel in rels))
new_df['bio_auntuncle_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_auntuncle for rel in rels))
new_df['bio_auntuncle_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_auntuncle for rel in rels))
new_df['marriage_auntuncle_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_auntuncle for rel in rels))
new_df['marriage_auntuncle_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_auntuncle for rel in rels))
new_df['social_auntuncle_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_auntuncle for rel in rels))
new_df['social_auntuncle_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_auntuncle for rel in rels))



# Siblings
new_df['any_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_sibling for rel in rels))
new_df['any_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_sibling for rel in rels))
new_df['bio_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_sibling for rel in rels))
new_df['bio_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_sibling for rel in rels))
new_df['step_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_sibling for rel in rels))
new_df['step_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_sibling for rel in rels))
new_df['social_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_sibling for rel in rels))
new_df['social_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_sibling for rel in rels))
new_df['in_law_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_sibling for rel in rels))
new_df['in_law_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_sibling for rel in rels))
new_df['social_in_law_sibling_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_sibling for rel in rels))
new_df['social_in_law_sibling_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_sibling for rel in rels))  


# Cousins 
new_df['any_cousin_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_cousin for rel in rels))
new_df['any_cousin_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_cousin for rel in rels))
new_df['cousin_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in cousin for rel in rels))
new_df['cousin_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in cousin for rel in rels))
new_df['marriage_cousin_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_cousin for rel in rels))
new_df['marriage_cousin_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_cousin for rel in rels))
new_df['social_cousin_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_cousin for rel in rels))
new_df['social_cousin_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_cousin for rel in rels))


# niece/nephew
new_df['any_nephniece_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_nephniece for rel in rels))
new_df['any_nephniece_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_nephniece for rel in rels))
new_df['bio_nephniece_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_nephniece for rel in rels))
new_df['bio_nephniece_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_nephniece for rel in rels))
new_df['marriage_nephniece_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_nephniece for rel in rels))
new_df['marriage_nephniece_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_nephniece for rel in rels))
new_df['social_nephniece_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_nephniece for rel in rels))
new_df['social_nephniece_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_nephniece for rel in rels))


# children --> should approx 0
new_df['any_ownchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ownchild for rel in rels))  
new_df['any_ownchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ownchild for rel in rels))
new_df['bio_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_child for rel in rels))
new_df['bio_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_child for rel in rels))
new_df['step_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_child for rel in rels))
new_df['step_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_child for rel in rels))
new_df['social_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_child for  rel in rels))
new_df['social_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_child for rel in rels))
new_df['foster_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_child for rel in rels))
new_df['foster_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_child for rel in rels))
new_df['in_law_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_child for rel in rels))
new_df['in_law_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in in_law_child for rel in rels))
new_df['social_in_law_child_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_child for rel in rels))
new_df['social_in_law_child_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_in_law_child for rel in rels))


# Grandchild / greatgrandchild --> should all be missing
new_df['any_grandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_grandchild for rel in rels))
new_df['any_grandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_grandchild for rel in rels))
new_df['any_greatgrandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_greatgrandchild for rel in rels))
new_df['any_greatgrandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_greatgrandchild for rel in rels))
new_df['any_ggchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ggchild for rel in rels))
new_df['any_ggchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_ggchild for rel in rels))
new_df['bio_grandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_grandchild for rel in rels))
new_df['bio_grandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_grandchild for rel in rels))
new_df['step_grandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_grandchild for rel in rels))
new_df['step_grandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_grandchild for rel in rels))
new_df['inlaw_grandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_grandchild for rel in rels))
new_df['inlaw_grandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_grandchild for rel in rels))
new_df['social_grandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_grandchild for rel in rels))
new_df['social_grandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_grandchild for rel in rels))
new_df['bio_greatgrandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_greatgrandchild for rel in rels))
new_df['bio_greatgrandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_greatgrandchild for rel in rels))
new_df['step_greatgrandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in step_greatgrandchild for rel in rels))
new_df['step_greatgrandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in step_greatgrandchild for rel in rels))
new_df['inlaw_greatgrandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_greatgrandchild for rel in rels))
new_df['inlaw_greatgrandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in inlaw_greatgrandchild for rel in rels))
new_df['social_greatgrandchild_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_greatgrandchild for rel in rels))
new_df['social_greatgrandchild_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_greatgrandchild for rel in rels))


# unknown 
new_df['unknown_rel_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in unknown for rel in rels))
new_df['unknown_rel_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in unknown for rel in rels))

# foster
new_df['foster_other_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_other for rel in rels))
new_df['foster_other_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in foster_other for rel in rels))


# other relative
new_df['any_relativeother_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in any_relativeother for rel in rels))
new_df['any_relativeother_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in any_relativeother for rel in rels))
new_df['bio_relativeother_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_relativeother for rel in rels))
new_df['bio_relativeother_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in bio_relativeother for rel in rels))
new_df['marriage_relativeother_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_relativeother for rel in rels))
new_df['marriage_relativeother_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in marriage_relativeother for rel in rels))
new_df['social_relativeother_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in social_relativeother for rel in rels))
new_df['social_relativeother_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in social_relativeother for rel in rels))

# nonrelative
new_df['nonrelative_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in nonrelative for rel in rels))
new_df['nonrelative_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in nonrelative for rel in rels))

# spouse/partner
new_df['spouse_partner_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in spouse_partner for rel in rels))
new_df['spouse_partner_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in spouse_partner for rel in rels))
new_df['spouse_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in spouse for rel in rels))
new_df['spouse_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in spouse for rel in rels))
new_df['partner_came'] = df['rel_came'].apply(lambda rels: isinstance(rels, list) and any(rel in partner for rel in rels))
new_df['partner_left'] = df['rel_left'].apply(lambda rels: isinstance(rels, list) and any(rel in partner for rel in rels))

In [20]:
# concat!
df = pd.concat([df, pd.DataFrame(new_df)], axis=1)

In [21]:
# Non-sibling adult came/left
new2 = {}
df = df.loc[:, ~df.columns.duplicated()].copy()

In [22]:
list5 = any_parent + any_gpar + any_ggpar + any_auntuncle + any_sibling + any_cousin + any_nephniece + any_ownchild + any_ggchild  + unknown + foster_other + any_relativeother + nonrelative + spouse_partner

new2['adult_came2'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in list5
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new2['adult_left2'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in list5
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

In [23]:
# non-sibling adult came/left

new2['nonsib_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel not in any_sibling
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new2['nonsib_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel not in any_sibling
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

# adult in generation above came/left
genabove = any_parent + any_gpar_ggpar + any_auntuncle + unknown + foster_other + any_relativeother + nonrelative

new2['genabove_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in genabove
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new2['genabove_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in genabove
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

# removing unknown, others, nonrelatives

genabove2 = any_parent + any_gpar_ggpar + any_auntuncle

new2['genabove2_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in genabove2
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new2['genabove2_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in genabove2
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


In [24]:
df = pd.concat([df, pd.DataFrame(new2)], axis=1)

In [25]:
new3 = {}
df = df.loc[:, ~df.columns.duplicated()].copy()

In [26]:

#  parent_adult_came/left

new3['parent_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_parent
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['parent_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_parent
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

# grandparent_adult_came/left

new3['grandparent_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_gpar
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['grandparent_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_gpar
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


# grand-grandparent_adult_came/left

new3['grandgrandparent_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ggpar
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['grandgrandparent_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ggpar
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


# aunt-uncle came/left

new3['auntuncle_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_auntuncle
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['auntuncle_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_auntuncle
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

# sibling came/left 

new3['sibling_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_sibling
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['sibling_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_sibling
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


# cousin came/left 

new3['cousin_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_cousin
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['cousin_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_cousin
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


# niece neph came/left

new3['niece_neph_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_nephniece
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['niece_neph_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_nephniece
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

#any_ownchild came/left

new3['any_ownchild_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ownchild
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['any_ownchild_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ownchild
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

#any_ggchild came/left

new3['any_ggchild_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ggchild
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['any_ggchild_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_ggchild
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


# unknown adult came/left

new3['unknown_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in unknown
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['unknown_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in unknown
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)


new3['foster_other_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in foster_other
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['foster_other_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in foster_other
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)
# non-relative adult came/left

new3['nonrelative_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in nonrelative
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['nonrelative_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in nonrelative
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

#  any_relativeother 
new3['any_relativeother_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_relativeother
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['any_relativeother_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in any_relativeother
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)

# spouse/partner came/left

new3['spousepartner_adult_came'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in spouse_partner
        for age, rel in zip(row['ages_came'], row['rel_came'])
    ) if isinstance(row['ages_came'], list) and isinstance(row['rel_came'], list) else False,
    axis=1
)

new3['spousepartner_adult_left'] = df.apply(
    lambda row: any(
        int(age) >= 18 and int(age) != 999 and rel in spouse_partner
        for age, rel in zip(row['ages_left'], row['rel_left'])
    ) if isinstance(row['ages_left'], list) and isinstance(row['rel_left'], list) else False,
    axis=1
)



In [27]:
df = pd.concat([df, pd.DataFrame(new3)], axis=1)

In [28]:
# output
df.to_csv(f"{output}/_hhr.csv", index=False)

In [29]:
df.columns.tolist()

['fam',
 'ID',
 'year',
 'fam_id_',
 'age_',
 'hhr_matrix',
 'rel_matrix',
 'age_matrix',
 'hhr_no_self',
 'ages_no_self',
 'rel_no_self',
 'sib_list',
 'par_list',
 'gpar_list',
 'sample_indiv_N',
 'sample_indiv_A',
 'sample_indiv_B',
 'why_left_survey',
 'year_left_survey',
 'hhr',
 'ages',
 'rel',
 'siblings',
 'parents',
 'grandparents',
 'hhr_prev',
 'rel_prev',
 'ages_prev',
 'IDs_left',
 'IDs_came',
 'is_first',
 'is_last',
 'ages_left',
 'ages_came',
 'rel_left',
 'rel_came',
 'adult_came',
 'adult_left',
 'child_came',
 'child_left',
 'unknown_came',
 'unknown_left',
 'rel_left_desc',
 'rel_came_desc',
 'parent_came',
 'parent_left',
 'nonbio_parent_came',
 'nonbio_parent_left',
 'bio_parent_came',
 'bio_parent_left',
 'step_parent_came',
 'step_parent_left',
 'social_parent_came',
 'social_parent_left',
 'in_law_parent_came',
 'in_law_parent_left',
 'foster_parent_came',
 'foster_parent_left',
 'social_in_law_parent_came',
 'social_in_law_parent_left',
 'any_gpar_ggpar_came',